# conduyt-pilot fine-tune (Phase 2, v5)

SFT fine-tune of `unsloth/Qwen2.5-Coder-1.5B-Instruct` with QLoRA + Unsloth on a Kaggle T4 instance (single T4 is correct here; Unsloth doesn't auto-shard a 1.5B QLoRA across two GPUs, so T4 x2 would just burn 2x quota for the same speed), then merge + export to GGUF (Q4_K_M / Q5_K_M / Q8_0). MLC conversion lives in `convert_mlc.ipynb` (different env).

**Dataset (v5)**: 453 training examples, 49 eval, stratified across 8 boards. Buckets cover conduyt-js single-turn and multi-turn, Arduino-vs-conduyt-js disambiguation (v3), hardware/electronics breadth (v4), and bare-metal ARM/RISC-V/STM32 LL/PlatformIO/FOC imports from `electron-rare/mascarade` MIT (v5). See `tracking/handoffs/` for per-version rationale.

**Hyperparameters (unchanged from v2)**: LoRA `r=32, alpha=64`, `lora_dropout=0`, `num_train_epochs=5`, `fp16=True`, effective batch 8. Bumping the dataset (v3, v4, v5) without touching the hyperparameters is intentional - the v3/v4/v5 wins come from contrastive examples and breadth, not from training tweaks.

**Target HF repos (v5)**: `virgilvox/conduyt-pilot-1.5b-v5-{lora,merged,gguf}`. Each major dataset version gets its own repo so prior runs stay archived for comparison.

Wall-clock budget: roughly 60 to 90 minutes for 453 examples x 5 epochs on a single T4 (training ~30-45 min, GGUF quantize ~10-15 min, HF upload ~10-15 min).


In [ ]:
# Pinned versions. Unsloth's API moves; pinning keeps this notebook reproducible.
# Source for these pins: Unsloth's own current Kaggle Qwen2.5-Coder reference notebook,
# cross-checked against unsloth==2026.4.8 requires_dist on PyPI.
#
# Notes:
#  - Kaggle T4 image ships with CUDA 12.x, torch >= 2.4. We let unsloth resolve
#    its own torch/xformers since the installer detects the runtime.
#  - `trl` is installed --no-deps to avoid pulling a transformers version that
#    conflicts with the unsloth pin.
#  - `datasets<4.4.0` is required by unsloth 2026.4.8.
#  - huggingface_hub is intentionally NOT pinned: transformers 4.56.2 caps it at
#    <1.0, while unsloth caps it at >=0.34.0. Let pip resolve.
#  - The fsspec / s3fs / gcsfs / bigframes warnings from Kaggle's pre-installed
#    packages are unrelated and safe to ignore for training.
#
!pip install -q --upgrade pip
!pip install -q unsloth==2026.4.8
!pip install -q --no-deps trl==0.22.2
!pip install -q transformers==4.56.2 \
                  peft==0.19.1 \
                  bitsandbytes==0.49.2 \
                  accelerate==1.13.0 \
                  datasets==4.3.0 \
                  wandb


In [ ]:
# Secrets. Kaggle's UserSecretsClient is the only way to access them at runtime.
# Required: HF_TOKEN (write scope) for pushing the adapter and GGUF.
# Optional: WANDB_API_KEY for run logging.
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

USE_WANDB = False
try:
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    import wandb  # noqa: F401  (defensive: if the secret exists but wandb wasn't installed, trainer init would crash mid-train)
    USE_WANDB = True
except Exception:
    pass
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("WANDB on:    ", USE_WANDB)


In [ ]:
# Load the dataset. The Kaggle Dataset is attached via the right-hand panel
# (Add Input). We auto-discover the mount path by looking for the directory
# that contains train.jsonl, so this works regardless of the slug Kaggle
# assigned (e.g. `conduyt-pilot-data`, `conduyt-pilot-data-v1`, etc).
import json, os, glob
from datasets import Dataset

INPUT_ROOT = "/kaggle/input"

candidates = glob.glob(f"{INPUT_ROOT}/**/train.jsonl", recursive=True)
if not candidates:
    available = os.listdir(INPUT_ROOT) if os.path.exists(INPUT_ROOT) else "(none)"
    raise FileNotFoundError(
        f"No train.jsonl found under {INPUT_ROOT}. Available inputs: {available}. "
        "Attach the conduyt-pilot-data dataset via the right-hand panel (Add Input)."
    )
DATA_DIR = os.path.dirname(candidates[0])
print(f"DATA_DIR = {DATA_DIR}")

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(f"{DATA_DIR}/train.jsonl")
eval_rows  = load_jsonl(f"{DATA_DIR}/eval.jsonl")

train_ds = Dataset.from_list(train_rows)
eval_ds  = Dataset.from_list(eval_rows)

print(f"train: {len(train_ds)}  eval: {len(eval_ds)}")
print("--- first train example ---")
print(json.dumps(train_rows[0], indent=2)[:800])


In [ ]:
# Load the base model with 4-bit QLoRA via Unsloth's FastLanguageModel.
# dtype=None lets Unsloth pick the best for the T4 (which is fp16-only; bf16 is
# unsupported on T4 silicon).
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 4096
MODEL_NAME     = "unsloth/Qwen2.5-Coder-1.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    load_in_4bit    = True,
    dtype           = None,
)
print(model)


In [ ]:
# LoRA config (unchanged across v2-v5). r=32, alpha=64 to give the LoRA more capacity
# Compared to v1's r=16, this gives more capacity to override the base
# model's Arduino prior. v3+ adds explicit Arduino-vs-conduyt-js contrast
# examples on top of this; the LoRA capacity remains the dominant lever.
#
# lora_dropout=0 is required for Unsloth's fast-patching kernels (~2-3x speedup).
# For datasets this small, dropout regularization is doing almost nothing anyway.
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 32,
    lora_alpha                 = 64,
    target_modules             = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout               = 0,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 3407,
)


In [ ]:
# Format with the Qwen2.5 chat template. Each example already has a `messages`
# field; apply_chat_template renders it into the model's expected token sequence.
def format_example(row):
    text = tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False,
    )
    return { "text": text }

train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds  = eval_ds.map( format_example, remove_columns=eval_ds.column_names)

print("--- formatted example (truncated) ---")
print(train_ds[0]["text"][:1200])


In [ ]:
# SFT trainer config. fp16=True is required on T4. `num_train_epochs` scales with
# dataset size, which is what we want here (a fixed `max_steps` would underfit
# bigger datasets later).
#
# v2-v5: 5 epochs (v1 was 3). With 453 train examples (v5) at effective
# batch 8, that's ~283 gradient passes per epoch x 5 = ~1415 total updates.
# Plenty for a 1.5B model with a 30-50 MB LoRA on this corpus size.
from trl import SFTTrainer, SFTConfig

config = SFTConfig(
    output_dir                  = "/kaggle/working/outputs",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps                = 10,
    num_train_epochs            = 5,
    learning_rate               = 2e-4,
    fp16                        = True,
    bf16                        = False,
    logging_steps               = 5,
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    lr_scheduler_type           = "linear",
    seed                        = 3407,
    save_strategy               = "epoch",
    eval_strategy               = "epoch",
    report_to                   = "wandb" if USE_WANDB else "none",
    max_seq_length              = MAX_SEQ_LENGTH,
    dataset_text_field          = "text",
    packing                     = False,
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    args          = config,
)


In [ ]:
import time
t0 = time.time()
stats = trainer.train()
elapsed = time.time() - t0
print(f"trained in {elapsed/60:.1f} min")
metrics = trainer.evaluate()
print("final eval:", metrics)


In [ ]:
# Save adapter locally and push to HF Hub.
ADAPTER_DIR  = "/kaggle/working/adapter"
ADAPTER_REPO = "virgilvox/conduyt-pilot-1.5b-v5-lora"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

model.push_to_hub(ADAPTER_REPO, token=os.environ["HF_TOKEN"])
tokenizer.push_to_hub(ADAPTER_REPO, token=os.environ["HF_TOKEN"])
print("adapter pushed to https://huggingface.co/" + ADAPTER_REPO)


In [ ]:
# Quick sanity inference. Five probes that each exercise a different bucket
# of the v5 corpus. Eyeball the outputs:
#   1. dont-mix: system says "no Arduino primitives in JS"; user asks a JS
#      task. Model should produce setTimeout, not delay().
#   2. side-by-side: system says "show both"; model should emit two labeled
#      code blocks (Arduino sketch + conduyt-js host).
#   3. hardware fundamentals: system is the v4 hardware advisor; user asks
#      Ohm's-law sized question. Model should give specific R + math.
#   4. debug walkthrough: system is the v4 debug advisor; user reports a
#      symptom. Model should rank causes most-likely first.
#   5. mascarade bare-metal: tests whether the v5 imports show up at
#      inference. Model should produce an assembly or C startup with vector
#      table, not Arduino sketch boilerplate.
FastLanguageModel.for_inference(model)

SYS_DONT_MIX = (
    "You write conduyt-js host code only. JavaScript and TypeScript do not "
    "have digitalWrite, pinMode, millis, delay, or Arduino-style map. Those "
    "are Arduino C++ primitives that compile into the microcontroller "
    "binary. If you find yourself reaching for one of those, stop and use "
    "the JavaScript or conduyt-js equivalent. Imports come from 'conduyt-js' "
    "and its subpaths. Output JavaScript or TypeScript only."
)
SYS_BOTH = (
    "You explain hardware tasks for two distinct toolchains. Arduino sketches "
    "are C++ that compiles to and runs on the microcontroller. conduyt-js is "
    "JavaScript or TypeScript that runs on a host computer (Node.js or "
    "browser) and talks to a microcontroller running CONDUYT firmware. When "
    "asked to show both, output two separate, clearly labeled code blocks. "
    "Never mix Arduino primitives into the JavaScript block."
)
SYS_HARDWARE = (
    "You are a hardware engineering and embedded systems advisor. Answer "
    "with specific component values, part numbers, pin assignments, and "
    "code where relevant. Cite typical values rather than vague "
    "descriptions. State assumptions explicitly when the question is "
    "ambiguous. Tone is direct and technical. Short over long."
)
SYS_DEBUG = (
    "You are a debugging assistant for embedded hardware. When given a "
    "symptom, list the most-likely-first causes with concrete diagnostic "
    "steps. Cite component checks, multimeter readings, oscilloscope or "
    "logic analyzer captures, and software state to inspect. Include the "
    "exact register, command, or code path where the bug usually hides."
)
SYS_EMBEDDED = (
    "You are an expert embedded systems engineer specializing in ARM "
    "Cortex-M (STM32, nRF52, Teensy, SAMD), ESP32/ESP-IDF, and RISC-V "
    "architectures. Provide complete, working code with explanations."
)

PROBES = [
    {
        "name": "dont-mix (v3)",
        "messages": [
            {"role": "system",  "content": SYS_DONT_MIX},
            {"role": "user",    "content": "Wait 250 ms, then write 1 to pin 13."},
        ],
    },
    {
        "name": "side-by-side (v3)",
        "messages": [
            {"role": "system",  "content": SYS_BOTH},
            {"role": "user",    "content": "Show both: blink an LED on D13 every 500 ms as an Arduino sketch and as conduyt-js host code over USB serial."},
        ],
    },
    {
        "name": "hardware fundamentals (v4)",
        "messages": [
            {"role": "system",  "content": SYS_HARDWARE},
            {"role": "user",    "content": "Current-limiting resistor for a 5 mm green LED driven by a 3.3 V GPIO at 10 mA. Show the math."},
        ],
    },
    {
        "name": "debug walkthrough (v4)",
        "messages": [
            {"role": "system",  "content": SYS_DEBUG},
            {"role": "user",    "content": "I2C scan returns nothing on an ESP32-S3 DevKitC-1. The sensor is wired and powered."},
        ],
    },
    {
        "name": "mascarade bare-metal (v5)",
        "messages": [
            {"role": "system",  "content": SYS_EMBEDDED},
            {"role": "user",    "content": "Write bare-metal ARM Cortex-M4 startup code in assembly with vector table and Reset_Handler. Target STM32F4xx."},
        ],
    },
]

for i, probe in enumerate(PROBES, 1):
    inputs = tokenizer.apply_chat_template(
        probe["messages"], tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=400, do_sample=False)
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"--- probe {i}: {probe['name']} ---")
    print("USER:", probe["messages"][-1]["content"])
    print("---")
    print(text)
    print()


In [ ]:
# Merge LoRA into the base weights, save full-precision merged model.
# Required for GGUF + MLC conversion downstream.
MERGED_DIR = "/kaggle/working/merged"

model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method = "merged_16bit",
)
print("merged model at", MERGED_DIR)


In [ ]:
# GGUF export at three quants. save_pretrained_gguf() vendors llama.cpp's
# convert + quantize binaries; expect a few minutes per quant.
GGUF_REPO = "virgilvox/conduyt-pilot-1.5b-v5-gguf"

for quant in ["q4_k_m", "q5_k_m", "q8_0"]:
    out_dir = f"/kaggle/working/gguf_{quant}"
    model.save_pretrained_gguf(
        out_dir,
        tokenizer,
        quantization_method = quant,
    )
    print(f"built {quant} -> {out_dir}")

# Push all three to one HF repo. We upload the .gguf files individually so the
# repo ends up with a flat layout (good for llama.cpp / Ollama / llamafile).
from huggingface_hub import HfApi, create_repo
import glob

api = HfApi(token=os.environ["HF_TOKEN"])
create_repo(GGUF_REPO, exist_ok=True, token=os.environ["HF_TOKEN"])

for path in sorted(glob.glob("/kaggle/working/gguf_*/**/*.gguf", recursive=True)):
    fname = path.split("/")[-1]
    api.upload_file(
        path_or_fileobj = path,
        path_in_repo    = fname,
        repo_id         = GGUF_REPO,
        repo_type       = "model",
    )
    print(f"pushed {fname}")

print("done. browse at https://huggingface.co/" + GGUF_REPO)


In [ ]:
# Push the merged 16-bit model to HF so convert_mlc.ipynb (running in a fresh
# kernel) can snapshot_download it. Without this, MLC conversion has nothing
# to ingest.
from huggingface_hub import HfApi, create_repo

MERGED_REPO = "virgilvox/conduyt-pilot-1.5b-v5-merged"
api = HfApi(token=os.environ["HF_TOKEN"])
create_repo(MERGED_REPO, exist_ok=True, token=os.environ["HF_TOKEN"], private=True)
api.upload_folder(
    folder_path = MERGED_DIR,
    repo_id     = MERGED_REPO,
    repo_type   = "model",
)
print("merged pushed to https://huggingface.co/" + MERGED_REPO)
print("now run convert_mlc.ipynb with MERGED_REPO =", MERGED_REPO)


## Next step: MLC conversion

MLC LLM uses a different toolchain (TVM-based) and pulls in different CUDA pins, so we keep it in a separate notebook to avoid env conflicts. Open `convert_mlc.ipynb` next; it pulls the merged model from `virgilvox/conduyt-pilot-1.5b-v5-merged` (just pushed in the cell above) via `snapshot_download`.
